**IMPORT LIBRARIES**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras

**LOAD DATASET**

In [2]:
link = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
data = pd.read_csv(link)
data = pd.DataFrame(data)

print(data.head())

                status  duration                            credit_history  \
0         ... < 100 DM         6   critical account/other credits existing   
1    0 <= ... < 200 DM        48  existing credits paid back duly till now   
2  no checking account        12   critical account/other credits existing   
3         ... < 100 DM        42  existing credits paid back duly till now   
4         ... < 100 DM        24           delay in paying off in the past   

               purpose  amount                     savings  \
0  domestic appliances    1169  unknown/no savings account   
1  domestic appliances    5951                ... < 100 DM   
2           retraining    2096                ... < 100 DM   
3     radio/television    7882                ... < 100 DM   
4            car (new)    4870                ... < 100 DM   

  employment_duration  installment_rate                  personal_status_sex  \
0      ... >= 7 years                 4                        male : single  

**CHECK DATA**

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   status                   1000 non-null   object
 1   duration                 1000 non-null   int64 
 2   credit_history           1000 non-null   object
 3   purpose                  1000 non-null   object
 4   amount                   1000 non-null   int64 
 5   savings                  1000 non-null   object
 6   employment_duration      1000 non-null   object
 7   installment_rate         1000 non-null   int64 
 8   personal_status_sex      1000 non-null   object
 9   other_debtors            1000 non-null   object
 10  present_residence        1000 non-null   int64 
 11  property                 1000 non-null   object
 12  age                      1000 non-null   int64 
 13  other_installment_plans  1000 non-null   object
 14  housing                  1000 non-null   

In [4]:
data.describe()

,duration,amount,installment_rate,present_residence,age,number_credits,people_liable,credit_risk
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000,0.700000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086,0.458487
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000,0.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000,0.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000,1.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000,1.000000


In [5]:
data['credit_risk'].unique()

array([1, 0])

In [6]:
data.shape

(1000, 21)

**Preprocessing (Convert categorical → numeric)**

In [7]:
data = pd.get_dummies(data , drop_first=True)
data = data

**Features and Target separation**

In [8]:
x= data.drop("credit_risk", axis=1)
y= data["credit_risk"]

**TRAIN TEST SPLIT**

In [9]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

**feature scaling**

In [10]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

**creating model**

In [11]:
from sklearn.utils import class_weight
import numpy as np

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {0: weights[0], 1: weights[1]}

In [13]:
model = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(x_train.shape[1],)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**compiling model**

In [14]:
model.compile(
   optimizer= "adam",
   loss= "binary_crossentropy",
   metrics=["accuracy"]
)

In [ ]:
**trining model**

In [15]:
history = model.fit(
    x_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(x_test, y_test),
    class_weight=class_weights
)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.6162 - loss: 0.7165 - val_accuracy: 0.5200 - val_loss: 0.7033
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6488 - loss: 0.6177 - val_accuracy: 0.6050 - val_loss: 0.6375
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6737 - loss: 0.6033 - val_accuracy: 0.6650 - val_loss: 0.6141
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6963 - loss: 0.5548 - val_accuracy: 0.6700 - val_loss: 0.5947
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7050 - loss: 0.5633 - val_accuracy: 0.6700 - val_loss: 0.5893
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7212 - loss: 0.5374 - val_accuracy: 0.6750 - val_loss: 0.5873
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7262 - loss: 0.5166 - val_accuracy: 0.6800 - val_loss: 0.5746
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7550 - loss: 0.5006 - val_accuracy: 0.6850 - val_loss

**accuracy**

In [16]:
loss , accuracy = model.evaluate(x_test,y_test)
print("accuracy", accuracy)
print("loss", loss)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7100 - loss: 0.7646  
accuracy 0.7099999785423279
loss 0.764637291431427


**making prediction**

In [17]:
y_pred=model.predict(x_test)
y_pred=(y_pred>0.5).astype(int)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


**Confusion Matrix**

In [18]:
cm = confusion_matrix(y_test,y_pred)
cm

array([[ 40,  19],
       [ 39, 102]])

In [19]:
print(confusion_matrix(y_test, y_pred))

[[ 40  19]
 [ 39 102]]


**CREATING A PICKLE FILE**

In [24]:


import pickle
from tensorflow import keras


model.save("model.keras")


pickle.dump(scaler, open("scaler.pkl", "wb"))



model = keras.models.load_model("model.h5")

scaler = pickle.load(open("scaler.pkl", "rb"))